# Trabalho final da disciplina de Processamento de Linguagem Natural. DC-UFSCar
## Recapitulação do Seminário 1 + Pré-processamento e Representação Textual

**Dataset:** BrScamsFacebook (~450 mensagens, desbalanceamento ≈83% Spam / 17% Ham)
**Tarefa:** Classificação binária de mensagens do Facebook em `Ham` (legítima) ou `Spam` (golpe/scam).

Este notebook cobre:
1. Carregamento do dataset (upload manual no Colab)
2. Conversão do problema multiclasse original em binário (Ham/Spam)
3. Limpeza e normalização de texto
4. Tokenização, remoção de stopwords (PT-BR) e stemming/lemmatização
5. Representação textual: Bag-of-Words (CountVectorizer) e TF-IDF (TfidfVectorizer)
6. Exportação dos dados pré-processados e dos vetores (.csv / .pkl) para uso das Pessoas 2, 3 e 4

> **Fonte do dataset:** BrScamsFacebook foi produzido por terceiros. Link de acesso usado pelo grupo:
> `https://www.kaggle.com/datasets/brunaassisgt/brscamsfacebook`


## 1. Instalação de dependências

In [ ]:
# Bibliotecas necessárias para o pipeline de PLN em português
!pip install -q nltk spacy unidecode scikit-learn pandas
!python -m spacy download pt_core_news_sm -q


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.0/13.0 MB 51.6 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('pt_core_news_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


## 2. Upload do dataset

Rode a célula abaixo e selecione o arquivo do **BrScamsFacebook** (.csv ou .xlsx) quando o botão de upload aparecer.

In [ ]:
from google.colab import files

print("Selecione o arquivo do dataset BrScamsFacebook para upload (.csv ou .xlsx):")
uploaded = files.upload()

# Captura automaticamente o nome do primeiro arquivo enviado
nome_arquivo = list(uploaded.keys())[0]
print(f"\nArquivo carregado com sucesso: {nome_arquivo}")


Selecione o arquivo do dataset BrScamsFacebook para upload (.csv ou .xlsx):


Saving BrScamsFacebook.xlsx to BrScamsFacebook (3).xlsx

Arquivo carregado com sucesso: BrScamsFacebook (3).xlsx


## 3. Carregamento e inspeção do dataset

In [ ]:
import pandas as pd
import numpy as np
import re
import unicodedata
import pickle

import nltk
nltk.download('stopwords')
nltk.download('rslp')
nltk.download('punkt')
nltk.download('punkt_tab')

from nltk.corpus import stopwords
from nltk.stem import RSLPStemmer
from nltk.tokenize import word_tokenize

import spacy
nlp = spacy.load('pt_core_news_sm')

# Leitura do arquivo
if nome_arquivo.endswith('.xlsx'):
    df = pd.read_excel(nome_arquivo)
else:
    df = pd.read_csv(nome_arquivo)

print("Dimensões:", df.shape)
print("Colunas:", df.columns.tolist())
df.head()


[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package rslp to /root/nltk_data...
[nltk_data]   Package rslp is already up-to-date!
[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


Dimensões: (450, 3)
Colunas: ['Mensagem', 'Categoria', 'Dataset']


,Mensagem,Categoria,Dataset
0,"""✨ 3 Sinais Espirituais de que o amor de vocês...",Golpes Baseados Em Relacionamento,Interno
1,"""✅Se você está passando por alguma dificuldade...",Golpes Baseados Em Relacionamento,Interno
2,Sou solteiro e procuro uma namorada para ser m...,Golpes Baseados Em Relacionamento,Interno
3,"""Eu sou Amanda. 🌹\nMentora espiritual e criado...",Golpes Baseados Em Relacionamento,Interno
4,"""Ótimo diia e feliz domingo pra vocês tudos me...",Golpes Baseados Em Relacionamento,Interno


In [ ]:
coluna_texto = 'Mensagem'    # coluna com o texto da mensagem
coluna_label = 'Categoria'   # coluna com o rótulo

df[coluna_label].value_counts()


,count
Categoria,
Golpes Baseados Em Relacionamento,75
Fraudes em Lojas Virtuais Falsas,75
Golpes de Desinformação Digital,75
Golpes de Ganho Financeiro Ilusório,75
Seguro,75
Ataques de Phishing e Roubo de Dados,75


## 4. Conversão multiclasse → binário (Ham/Spam)

Reaproveitando a lógica do Relatório 1: os rótulos originais (multiclasse) são agrupados em duas classes,
`Ham` (mensagens legítimas) e `Spam` (golpes/scams de qualquer tipo).

In [ ]:
rotulos_ham = ['seguro']

def mapear_binario(rotulo):
    rotulo = str(rotulo).strip().lower()
    return 'Ham' if rotulo in rotulos_ham else 'Spam'

df['classe_binaria'] = df[coluna_label].apply(mapear_binario)

print(df['classe_binaria'].value_counts())
print(df['classe_binaria'].value_counts(normalize=True).mul(100).round(1))


classe_binaria
Spam    375
Ham      75
Name: count, dtype: int64
classe_binaria
Spam    83.3
Ham     16.7
Name: proportion, dtype: float64


## 5. Limpeza e normalização de texto

Remoção de URLs, menções (@usuario), emojis e caracteres especiais, além de normalização
(caixa baixa e remoção de acentos).

In [ ]:
def remover_acentos(texto):
    texto = unicodedata.normalize('NFKD', texto)
    return ''.join(c for c in texto if not unicodedata.combining(c))

def limpar_texto(texto):
    texto = str(texto).lower()
    texto = re.sub(r'http\S+|www\.\S+', ' ', texto)                     # URLs
    texto = re.sub(r'@\w+', ' ', texto)                                 # menções
    texto = re.sub(r'[\U0001F300-\U0001FAFF\U00002600-\U000027BF]', ' ', texto)  # emojis
    texto = remover_acentos(texto)                                     # remoção de acentos
    texto = re.sub(r'[^a-z\s]', ' ', texto)                             # números/pontuação/caracteres especiais
    texto = re.sub(r'\s+', ' ', texto).strip()                          # espaços extras
    return texto

df['texto_limpo'] = df[coluna_texto].apply(limpar_texto)
df[[coluna_texto, 'texto_limpo']].head()


,Mensagem,texto_limpo
0,"""✨ 3 Sinais Espirituais de que o amor de vocês...",sinais espirituais de que o amor de voces tem ...
1,"""✅Se você está passando por alguma dificuldade...",se voce esta passando por alguma dificuldade n...
2,Sou solteiro e procuro uma namorada para ser m...,sou solteiro e procuro uma namorada para ser m...
3,"""Eu sou Amanda. 🌹\nMentora espiritual e criado...",eu sou amanda mentora espiritual e criadora do...
4,"""Ótimo diia e feliz domingo pra vocês tudos me...",otimo diia e feliz domingo pra voces tudos meu...


## 6. Tokenização, stopwords e stemming/lemmatização

- **Tokenização + stopwords + stemming:** NLTK (`word_tokenize`, `RSLPStemmer`)
- **Lemmatização:** spaCy (`pt_core_news_sm`)

As duas versões são geradas para que o grupo possa comparar qual funciona melhor nos modelos.

In [ ]:
stopwords_pt = set(stopwords.words('portuguese'))
stemmer = RSLPStemmer()

def tokenizar_e_stemizar(texto):
    tokens = word_tokenize(texto, language='portuguese')
    tokens = [t for t in tokens if t not in stopwords_pt and len(t) > 1]
    stems = [stemmer.stem(t) for t in tokens]
    return tokens, stems

def lematizar(texto):
    doc = nlp(texto)
    lemas = [tok.lemma_ for tok in doc if tok.text not in stopwords_pt and not tok.is_punct and len(tok.text) > 1]
    return lemas

df['tokens'], df['stems'] = zip(*df['texto_limpo'].apply(tokenizar_e_stemizar))
df['lemas'] = df['texto_limpo'].apply(lematizar)

df['texto_stem'] = df['stems'].apply(' '.join)
df['texto_lema'] = df['lemas'].apply(' '.join)

df[['texto_limpo', 'texto_stem', 'texto_lema']].head()


,texto_limpo,texto_stem,texto_lema
0,sinais espirituais de que o amor de voces tem ...,sinal espirit am voc futur voc sent univers te...,sinal espiritual amor voce futuro voce sentido...
1,se voce esta passando por alguma dificuldade n...,voc pass algum dificuldad relacion clic link b...,voce passar algum dificuldade relacionamento c...
2,sou solteiro e procuro uma namorada para ser m...,solt procur namor companh sent so,solteiro procuro namorar companheira sentir so...
3,eu sou amanda mentora espiritual e criadora do...,amand men espirit cri rit supr unia amor ajud ...,amando mentora espiritual criadora ritual Supr...
4,otimo diia e feliz domingo pra voces tudos meu...,otim dii feliz doming pra voc tud am hom arab ...,otimo diia feliz domingo pra voce tur amor hom...


Comparar o tamanho do texto gerado por cada abordagem:

In [ ]:
vocab_stem = set(' '.join(df['texto_stem']).split())
vocab_lema = set(' '.join(df['texto_lema']).split())

print("Vocabulário único (stem):", len(vocab_stem))
print("Vocabulário único (lema):", len(vocab_lema))

Vocabulário único (stem): 3498
Vocabulário único (lema): 4336


## 7. Representação textual: Bag-of-Words e TF-IDF

Ambas as representações são geradas a partir do texto lematizado (`texto_lema`).
O grupo optou por trabalhar com lema.

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

texto_final = df['texto_lema']  # alternativa: df['texto_stem']

bow_vectorizer = CountVectorizer(max_features=3000)
X_bow = bow_vectorizer.fit_transform(texto_final)

tfidf_vectorizer = TfidfVectorizer(max_features=3000)
X_tfidf = tfidf_vectorizer.fit_transform(texto_final)

print("Bag-of-Words:", X_bow.shape)
print("TF-IDF:      ", X_tfidf.shape)


Bag-of-Words: (450, 3000)
TF-IDF:       (450, 3000)


## 8. Exportação dos dados pré-processados e dos vetores

Arquivos gerados para uso dos próximos membros:
- `brscams_preprocessado.csv`: dataset completo com texto limpo, tokens, stems, lemas e rótulo binário
- `bow_vectorizer.pkl` / `X_bow.pkl`: vetorizador e matriz Bag-of-Words
- `tfidf_vectorizer.pkl` / `X_tfidf.pkl`: vetorizador e matriz TF-IDF
- `labels.pkl`: rótulos binários (Ham/Spam)

In [ ]:
df.to_csv('brscams_preprocessado.csv', index=False)

with open('bow_vectorizer.pkl', 'wb') as f:
    pickle.dump(bow_vectorizer, f)
with open('X_bow.pkl', 'wb') as f:
    pickle.dump(X_bow, f)

with open('tfidf_vectorizer.pkl', 'wb') as f:
    pickle.dump(tfidf_vectorizer, f)
with open('X_tfidf.pkl', 'wb') as f:
    pickle.dump(X_tfidf, f)

with open('labels.pkl', 'wb') as f:
    pickle.dump(df['classe_binaria'], f)

print("Arquivos exportados com sucesso!")


Arquivos exportados com sucesso!


In [ ]:
#baixar os arquivos gerados para o seu computador
from google.colab import files

for nome in ['brscams_preprocessado.csv', 'bow_vectorizer.pkl', 'X_bow.pkl',
             'tfidf_vectorizer.pkl', 'X_tfidf.pkl', 'labels.pkl']:
    files.download(nome)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## 9. Notas para o relatório

**Dificuldades encontradas (preencher com as observações reais do grupo):**
- Textos curtos, o que aumenta a esparsidade das matrizes BoW/TF-IDF
- Uso de gírias e abreviações típicas de redes sociais, nem sempre cobertas pelas stopwords padrão
- Desbalanceamento das classes (≈83% Spam / 17% Ham), que deve ser considerado na etapa de modelagem
- Erros de digitação e variações ortográficas que dificultam a normalização

**Próximos passos:** os arquivos exportados (`.csv` e `.pkl`) devem ser subidos junto com este notebook no repositório do grupo em https://github.com/PLN-UFSCar, para uso das Pessoas 2, 3 e 4.